In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/iitg-ai-code-semantics-similarity-challenge/sample_submission.csv
/kaggle/input/competitions/iitg-ai-code-semantics-similarity-challenge/test.jsonl
/kaggle/input/competitions/iitg-ai-code-semantics-similarity-challenge/train.jsonl
/kaggle/input/competitions/iitg-ai-code-semantics-similarity-challenge/train_small.jsonl


In [2]:
!pip install -U transformers accelerate bitsandbytes>=0.46.1

In [3]:
# SmolLM-1.7B + QLoRA

# 1. IMPORTS
import os
import numpy as np
import pandas as pd
import torch


os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use a single GPU for stable 4-bit QLoRA training

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# 2. CONFIGURATION
SEED = 42
set_seed(SEED)

MODEL_NAME = "HuggingFaceTB/SmolLM-1.7B"

TRAIN_PATH = "/kaggle/input/competitions/iitg-ai-code-semantics-similarity-challenge/train_small.jsonl"
TEST_PATH = "/kaggle/input/competitions/iitg-ai-code-semantics-similarity-challenge/test.jsonl"
OUTPUT_DIR = "/kaggle/working/smollm_code_similarity"

MAX_LENGTH = 512

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
GRAD_ACCUMULATION = 8

NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 100

# LoRA Hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

In [4]:
# 4. LOAD AND SPLIT DATA
print("LOADING DATA...")

train_df = pd.read_json(TRAIN_PATH, lines=True)
test_df = pd.read_json(TEST_PATH, lines=True)

train_df = train_df.dropna(subset=["func1", "func2", "label"])
train_df["label"] = train_df["label"].apply(lambda x: 1 if str(x).lower() == "true" or x == 1 else 0)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)} | Test samples: {len(test_df)}")

train_dataset = Dataset.from_pandas(train_df[["func1", "func2", "label"]])
val_dataset = Dataset.from_pandas(val_df[["func1", "func2", "label"]])
test_dataset = Dataset.from_pandas(test_df[["id", "func1", "func2"]])

# 5. TOKENIZATION
print("TOKENIZING DATA...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def build_text(func1, func2):
    return f"### CODE 1\n{func1}\n\n### CODE 2\n{func2}\n\n### TASK\nDetermine whether CODE 1 and CODE 2 are semantically equivalent."

def tokenize_batch(batch):
    texts = [build_text(f1, f2) for f1, f2 in zip(batch["func1"], batch["func2"])]
    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

train_tokenized = train_dataset.map(
    tokenize_batch, batched=True, remove_columns=["func1", "func2"], desc="Tokenizing Train"
)
val_tokenized = val_dataset.map(
    tokenize_batch, batched=True, remove_columns=["func1", "func2"], desc="Tokenizing Validation"
)
test_tokenized = test_dataset.map(
    tokenize_batch, batched=True, remove_columns=["id", "func1", "func2"], desc="Tokenizing Test"
)

LOADING DATA...
Train samples: 450000 | Val samples: 50000 | Test samples: 5000
TOKENIZING DATA...


config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing Train:   0%|          | 0/450000 [00:00<?, ? examples/s]

Tokenizing Validation:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenizing Test:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [5]:
# 6. MODEL SETUP (4-BIT QLORA)
print("LOADING MODEL & QLORA...")

torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    quantization_config=bnb_config,
    device_map={"": 0},
)

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = False

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    modules_to_save=["score"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

LOADING MODEL & QLORA...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: HuggingFaceTB/SmolLM-1.7B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 6,295,552 || all params: 1,717,676,032 || trainable%: 0.3665


In [6]:
# 7. METRICS & TRAINING ARGUMENTS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1": f1_score(labels, preds, zero_division=0),
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0)
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=2000,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

In [ ]:
# 8. TRAINER INITIALIZATION & TRAINING

trainer = Trainer(
    model=model,  # Passes initialized PEFT model
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, padding=True, pad_to_multiple_of=8),
    compute_metrics=compute_metrics,
)

print("STARTING TRAINING...")
trainer.train()

------------------------------------------------------------
STARTING TRAINING...


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss,F1,Accuracy,Precision,Recall
500,1.819163,0.232892,0.898516,0.902500,0.870237,0.928694
1000,1.352334,0.166625,0.923561,0.931540,0.959894,0.889879
1500,1.191190,0.159814,0.933476,0.937840,0.928626,0.938377
2000,1.272665,0.144920,0.936147,0.941060,0.942745,0.929641


TrainOutput(global_step=2000, training_loss=1.9060810260772705, metrics={'train_runtime': 38670.7957, 'train_samples_per_second': 0.827, 'train_steps_per_second': 0.052, 'total_flos': 1.297777683384238e+17, 'train_loss': 1.9060810260772705, 'epoch': 0.07111111111111111})

In [8]:
# 9. SUBMISSION GENERATION
torch.cuda.empty_cache()

print("GENERATING PREDICTIONS...")

predictions = trainer.predict(test_tokenized)
test_pred = np.argmax(predictions.predictions, axis=-1)

submission = pd.DataFrame({
    "id": test_df["id"],
    "label": test_pred.astype(bool),
})

submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)

print("Submission file saved.")

GENERATING PREDICTIONS...


Submission file saved.
